# NFL Big Data Bowl 2026 — Spatiotemporal Transformer

Player trajectory prediction during the ball-in-air phase, modeling all 22 players plus the ball as a unified token graph.

## Approach

Each play is represented as a sequence of frames where every token corresponds to a player or the ball-landing reference. The target player occupies slot 0, with remaining players and a ball-landing token filling out the spatial dimension. From a 1-second pre-throw context window (10 frames at 10 Hz), the model predicts cumulative `(dx, dy)` deltas for up to 94 future frames.

The architecture stacks **axial attention blocks** that alternate between *temporal* attention (each player across time) and *spatial* attention (all players within a frame). Spatial attention is augmented with a **distance- and velocity-aware edge messaging** layer, giving the network an explicit prior over which player pairs should interact. Coordinates are unified to a single play direction during training and inverted back at inference time.

## Training

Cross-validation is performed with **GroupKFold by game_id** across **6 seeds × 5 folds**, producing 30 models for ensemble. Loss is a target-only **temporal Huber** with time-decay weighting, augmented by light physics regularizers (field boundary, per-frame speed cap, jerk smoothness). Each fold fits its own `StandardScaler`, and the resulting bundle (models + scalers + metadata) is consumed unchanged by the inference pipeline.

## Inference

The Kaggle inference gateway loads the bundle once and serves prediction batches lazily. Each batch passes through the full feature pipeline, ensemble averaging across all 30 models, and **8-step test-time augmentation** with small Gaussian noise on standardized channels. Final predictions are clipped to the field, inverted to the original play direction, and emitted in `(x, y)` form.


## Quick Note

If you want to reproduce the full result, restore the parameters to their original values. I re-organized the code with AI assistance before publishing, so some configs were lowered to speed up the verification run.

You may also need to adjust `BATCH_SIZE` depending on your GPU. I didn't train all seeds in a single Kaggle session — the last two were added in a separate run, so be mindful of the 9-hour notebook runtime limit.

**Original parameters:**
```
SEED          = 42
SEEDS         = [42, 44, 46, 31, 32, 28]
N_FOLDS       = 5
BATCH_SIZE    = 64
EPOCHS        = 200
PATIENCE      = 30
```



# Setup & Config

In [ ]:
import os
import json
import pickle
import random
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")


# --- NFL field constants ---
FIELD_LENGTH = 120.0
FIELD_WIDTH  = 53.3
FPS          = 10.0

# --- Sequence structure ---
MAX_PLAYERS = 22
BALL_ID     = -9999

# --- Role → type_id (model embedding) ---
# 0=Other, 1=TargetedReceiver, 2=DefensiveCoverage, 3=Passer, 4=Ball
ROLE_TO_TYPE = {
    "Targeted Receiver":  1,
    "Defensive Coverage": 2,
    "Passer":             3,
}
NUM_TYPES = 5

BAD_PLAYS = {(2023091100, 3167)}


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


class Config:
    # Paths
    DATA_DIR = Path(os.environ.get(
    "DATA_DIR",
    "/kaggle/input/competitions/nfl-big-data-bowl-2026-prediction/"
))
    OUTPUT_DIR            = Path("./outputs")
    MODEL_BUNDLE_DIR_TRAIN = OUTPUT_DIR / "bundle"

    _env_bundle = os.environ.get("MODEL_BUNDLE_DIR")
    MODEL_BUNDLE_DIR_SUB = (Path(_env_bundle) if _env_bundle
                            else Path("/kaggle/input/6seedens/outputs/bundle"))

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Training
    SEED          = 42
    SEEDS         = [42]
    N_FOLDS       = 2
    BATCH_SIZE    = 64
    EPOCHS        = 3
    PATIENCE      = 99
    LEARNING_RATE = 1e-3

    # Sequence
    WINDOW_SIZE        = 10
    MAX_FUTURE_HORIZON = 94

    # Feature radius (opponent features)
    NEIGHBOR_RADIUS = 12.0

    # Model
    HIDDEN_DIM = 128

    # Loss physics regularizers
    W_BOUNDARY    = 1e-4
    W_SPEED       = 1e-4
    W_JERK        = 1e-4
    MAX_SPEED_YPS = 12.0

    @classmethod
    def setup_dirs(cls):
        cls.OUTPUT_DIR.mkdir(exist_ok=True, parents=True)


TRAIN = int(os.environ.get("TRAIN", "1"))

set_seed(Config.SEED)
Config.setup_dirs()

print(f"Device: {Config.DEVICE}")
print(f"TRAIN mode: {TRAIN}")

# Field Geometry & Direction Unification

In [ ]:
def wrap_angle_deg(s):
    return ((s + 180.0) % 360.0) - 180.0


def build_play_direction_map(df: pd.DataFrame) -> pd.Series:
    return (df[['game_id', 'play_id', 'play_direction']]
            .drop_duplicates()
            .set_index(['game_id', 'play_id'])['play_direction'])


def unify_left_direction(df: pd.DataFrame) -> pd.DataFrame:
    if 'play_direction' not in df.columns:
        return df

    df = df.copy()
    right = df['play_direction'].eq('right')

    if 'x' in df.columns:
        df.loc[right, 'x'] = FIELD_LENGTH - df.loc[right, 'x']
    if 'y' in df.columns:
        df.loc[right, 'y'] = FIELD_WIDTH - df.loc[right, 'y']

    for col in ('dir', 'o'):
        if col in df.columns:
            df.loc[right, col] = (df.loc[right, col].astype(float) + 180.0) % 360.0

    if 'ball_land_x' in df.columns:
        df.loc[right, 'ball_land_x'] = FIELD_LENGTH - df.loc[right, 'ball_land_x']
    if 'ball_land_y' in df.columns:
        df.loc[right, 'ball_land_y'] = FIELD_WIDTH - df.loc[right, 'ball_land_y']

    return df


def apply_direction_to_df(df: pd.DataFrame, dir_map: pd.Series) -> pd.DataFrame:
    if 'play_direction' not in df.columns:
        dirm = dir_map.reset_index()
        df = df.merge(dirm, on=['game_id', 'play_id'], how='left', validate='many_to_one')
    return unify_left_direction(df)


def invert_to_original_direction(x_u: float, y_u: float, play_dir_right: bool):
    if not play_dir_right:
        return float(x_u), float(y_u)
    return float(FIELD_LENGTH - x_u), float(FIELD_WIDTH - y_u)

# Kinematics & Data Cleaning

In [ ]:
def ensure_basic_kinematics(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if 'dir' not in df.columns:
        df['dir'] = 0.0
    if 's' not in df.columns:
        df['s'] = 0.0
    if 'a' not in df.columns:
        df['a'] = 0.0

    rad = np.deg2rad(df['dir'].fillna(0.0).astype('float32'))

    if 'velocity_x' not in df.columns or 'velocity_y' not in df.columns:
        df['velocity_x'] = df['s'].astype('float32') * np.cos(rad)
        df['velocity_y'] = df['s'].astype('float32') * np.sin(rad)

    if 'acceleration_x' not in df.columns or 'acceleration_y' not in df.columns:
        df['acceleration_x'] = df['a'].astype('float32') * np.cos(rad)
        df['acceleration_y'] = df['a'].astype('float32') * np.sin(rad)

    if 'player_side' not in df.columns:
        df['player_side'] = 'Defense'

    return df


def drop_bad_plays(df: pd.DataFrame) -> pd.DataFrame:
    if not {'game_id', 'play_id'}.issubset(df.columns):
        return df

    key = df[['game_id', 'play_id']].apply(tuple, axis=1)
    kept = ~key.isin(BAD_PLAYS)

    if kept.sum() != len(df):
        n_dropped = int((~kept).sum())
        print(f"[clean] dropped {n_dropped} rows from {len(df)} due to BAD_PLAYS")

    return df[kept].reset_index(drop=True)

# Simple Feature Functions

In [ ]:
GROUP_COLS = ['game_id', 'play_id', 'nfl_id']


def add_role_and_ball_features(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    df = df.copy()

    df['is_offense']  = (df['player_side'] == 'Offense').astype('int8')
    df['is_defense']  = (df['player_side'] == 'Defense').astype('int8')
    df['is_receiver'] = (df.get('player_role', '') == 'Targeted Receiver').astype('int8')
    df['is_coverage'] = (df.get('player_role', '') == 'Defensive Coverage').astype('int8')
    df['is_passer']   = (df.get('player_role', '') == 'Passer').astype('int8')

    cols = [
        'x', 'y', 's', 'a', 'dir', 'frame_id',
        'velocity_x', 'velocity_y', 'acceleration_x', 'acceleration_y',
        'is_offense', 'is_defense', 'is_receiver', 'is_coverage', 'is_passer',
    ]

    if {'ball_land_x', 'ball_land_y'}.issubset(df.columns):
        bdx = df['ball_land_x'] - df['x']
        bdy = df['ball_land_y'] - df['y']
        dist = np.hypot(bdx, bdy)
        df['distance_to_ball'] = dist
        inv = 1.0 / (dist + 1e-6)
        df['ball_direction_x'] = bdx * inv
        df['ball_direction_y'] = bdy * inv
        cols += ['distance_to_ball', 'ball_direction_x', 'ball_direction_y']

    return df, cols


def add_time_features(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    df = df.copy()
    frames_elapsed = df.groupby(GROUP_COLS).cumcount()
    df['normalized_time'] = frames_elapsed / (
        frames_elapsed.groupby([df['game_id'], df['play_id'], df['nfl_id']]).transform('max') + 1e-9
    )
    return df, ['normalized_time']


def add_alignment_and_change_features(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    df = df.copy()
    cols = []

    if {'ball_direction_x', 'ball_direction_y', 'velocity_x', 'velocity_y'}.issubset(df.columns):
        df['velocity_alignment'] = (
            df['velocity_x'] * df['ball_direction_x'] +
            df['velocity_y'] * df['ball_direction_y']
        )
        df['velocity_perpendicular'] = (
            df['velocity_x'] * (-df['ball_direction_y']) +
            df['velocity_y'] * df['ball_direction_x']
        )
        cols += ['velocity_alignment', 'velocity_perpendicular']

        if {'acceleration_x', 'acceleration_y'}.issubset(df.columns):
            df['accel_alignment'] = (
                df['acceleration_x'] * df['ball_direction_x'] +
                df['acceleration_y'] * df['ball_direction_y']
            )
            cols.append('accel_alignment')

    if 'velocity_x' in df.columns:
        df['velocity_x_change'] = df.groupby(GROUP_COLS)['velocity_x'].diff().fillna(0.0)
        df['velocity_y_change'] = df.groupby(GROUP_COLS)['velocity_y'].diff().fillna(0.0)
        df['speed_change']      = df.groupby(GROUP_COLS)['s'].diff().fillna(0.0)
        ddir = df.groupby(GROUP_COLS)['dir'].diff().fillna(0.0)
        df['direction_change']  = wrap_angle_deg(ddir)
        cols += ['velocity_x_change', 'velocity_y_change', 'speed_change', 'direction_change']

    return df, cols


def add_field_position_features(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    df = df.copy()
    dist_left  = df['y']
    dist_right = FIELD_WIDTH - df['y']
    df['dist_from_sideline'] = np.minimum(dist_left, dist_right)
    df['dist_from_endzone']  = np.minimum(df['x'], FIELD_LENGTH - df['x'])
    return df, ['dist_from_sideline', 'dist_from_endzone']

# Opponent & Geo Goal Features

In [ ]:
def _compute_opponent_features_lastframe(df: pd.DataFrame,
                                         radius: float = 12.0,
                                         speed_eps: float = 0.5) -> pd.DataFrame:
    last = (df.sort_values(GROUP_COLS + ['frame_id'])
              .groupby(GROUP_COLS, as_index=False)
              .tail(1)
              .reset_index(drop=True))

    out_rows = []

    for (gid, pid), g in tqdm(last.groupby(['game_id', 'play_id']),
                               desc="opponent features",
                               leave=False,
                               bar_format="{l_bar}{bar:10}{r_bar}"):
        if len(g) <= 1:
            continue

        pos = g[['x', 'y']].values.astype('float32')
        vel = g[['velocity_x', 'velocity_y']].values.astype('float32')
        side_off = (g['player_side'].values == 'Offense')
        roles = g.get('player_role', pd.Series(['Other'] * len(g))).fillna('Other').values
        nfl_ids = g['nfl_id'].values

        rec_mask = np.isin(roles, ['Targeted Receiver', 'Other Route Runner'])
        rec_idx = np.where(rec_mask)[0]

        for i in range(len(g)):
            p_i = pos[i]
            v_i = vel[i]
            opp_mask = (side_off != side_off[i])
            if not opp_mask.any():
                continue

            opp_idx = np.where(opp_mask)[0]
            P_opp = pos[opp_idx]
            V_opp = vel[opp_idx]

            dxy = P_opp - p_i
            D = np.sqrt((dxy ** 2).sum(axis=1)) + 1e-6
            j_local = int(np.argmin(D))
            dmin = float(D[j_local])

            row = {'game_id': gid, 'play_id': pid, 'nfl_id': int(nfl_ids[i])}

            if dmin > radius:
                row['opp_dmin'] = radius
                row['opp_close_rate'] = 0.0
                row['opp_leverage'] = 0
            else:
                j_global = opp_idx[j_local]

                r = p_i - pos[j_global]
                u = r / (np.linalg.norm(r) + 1e-6)
                v_rel = V_opp[j_local] - v_i
                close = float(np.clip(np.dot(v_rel, u), -10.0, 10.0))

                speed = float(np.linalg.norm(v_i))
                to_opp = pos[j_global] - p_i
                cross_z = to_opp[0] * v_i[1] - to_opp[1] * v_i[0]
                leverage = int(np.sign(cross_z)) if speed > speed_eps else 0

                row['opp_dmin'] = dmin
                row['opp_close_rate'] = close
                row['opp_leverage'] = leverage

            if roles[i] == 'Defensive Coverage' and rec_mask.any():
                d_wr = np.sqrt(((pos[rec_idx] - p_i) ** 2).sum(axis=1))
                k_local = int(np.argmin(d_wr))
                k_global = rec_idx[k_local]

                row['mirror_wr_dist']  = float(d_wr[k_local])
                row['mirror_offset_x'] = float(p_i[0] - pos[k_global, 0])
                row['mirror_offset_y'] = float(p_i[1] - pos[k_global, 1])
                row['mirror_wr_vx']    = float(vel[k_global, 0])
                row['mirror_wr_vy']    = float(vel[k_global, 1])

                to_me = p_i - pos[k_global]
                u_wd = to_me / (np.linalg.norm(to_me) + 1e-6)
                row['closing_speed'] = float(((v_i - vel[k_global]) * u_wd).sum())

            out_rows.append(row)

    if not out_rows:
        return pd.DataFrame()

    return pd.DataFrame(out_rows).drop_duplicates(GROUP_COLS)


def add_opponent_features(df: pd.DataFrame, radius: float = 12.0) -> tuple[pd.DataFrame, list[str]]:
    last_frame = _compute_opponent_features_lastframe(df, radius=radius)

    if last_frame.empty:
        return df, []

    df = df.merge(last_frame, on=GROUP_COLS, how='left')

    df['opp_dmin']       = df['opp_dmin'].fillna(radius)
    df['opp_close_rate'] = df['opp_close_rate'].fillna(0.0)
    df['opp_leverage']   = df['opp_leverage'].fillna(0).astype('int8')

    cols = ['opp_dmin', 'opp_close_rate', 'opp_leverage']

    mirror_cols = ['mirror_wr_dist', 'mirror_offset_x', 'mirror_offset_y',
                   'mirror_wr_vx', 'mirror_wr_vy', 'closing_speed']
    for c in mirror_cols:
        if c in df.columns:
            cols.append(c)

    return df, cols


def add_geo_goal_features(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    df = df.copy()

    if 'num_frames_output' in df.columns:
        t_total = df['num_frames_output'].astype('float32') / FPS
    else:
        t_total = pd.Series(3.0, index=df.index, dtype='float32')

    t_total = t_total.clip(0.5, 10.0)
    df['time_to_endpoint'] = t_total

    df['geo_endpoint_x'] = df['x'] + df['velocity_x'] * t_total
    df['geo_endpoint_y'] = df['y'] + df['velocity_y'] * t_total

    if {'ball_land_x', 'ball_land_y', 'player_role'}.issubset(df.columns):
        rec = (df['player_role'] == 'Targeted Receiver')
        df.loc[rec, 'geo_endpoint_x'] = df.loc[rec, 'ball_land_x']
        df.loc[rec, 'geo_endpoint_y'] = df.loc[rec, 'ball_land_y']

    if {'player_role', 'mirror_wr_dist', 'mirror_offset_x', 'mirror_offset_y',
        'ball_land_x', 'ball_land_y'}.issubset(df.columns):
        def_mask = (df['player_role'] == 'Defensive Coverage')
        has_mirror = def_mask & (df['mirror_wr_dist'].fillna(1e9) < 15.0)
        df.loc[has_mirror, 'geo_endpoint_x'] = (
            df.loc[has_mirror, 'ball_land_x'] + df['mirror_offset_x'].fillna(0.0)
        )
        df.loc[has_mirror, 'geo_endpoint_y'] = (
            df.loc[has_mirror, 'ball_land_y'] + df['mirror_offset_y'].fillna(0.0)
        )

    df['geo_endpoint_x'] = df['geo_endpoint_x'].clip(0.0, FIELD_LENGTH)
    df['geo_endpoint_y'] = df['geo_endpoint_y'].clip(0.0, FIELD_WIDTH)

    df['geo_vector_x'] = df['geo_endpoint_x'] - df['x']
    df['geo_vector_y'] = df['geo_endpoint_y'] - df['y']
    df['geo_distance'] = np.hypot(df['geo_vector_x'], df['geo_vector_y'])

    t = t_total + 1e-6
    df['geo_required_vx'] = df['geo_vector_x'] / t
    df['geo_required_vy'] = df['geo_vector_y'] / t

    df['geo_velocity_error_x'] = df['geo_required_vx'] - df['velocity_x']
    df['geo_velocity_error_y'] = df['geo_required_vy'] - df['velocity_y']
    df['geo_velocity_error']   = np.hypot(df['geo_velocity_error_x'], df['geo_velocity_error_y'])

    t_sq = t * t
    df['geo_required_ax'] = (2.0 * df['geo_vector_x'] / t_sq).clip(-10, 10)
    df['geo_required_ay'] = (2.0 * df['geo_vector_y'] / t_sq).clip(-10, 10)

    vel_mag = np.hypot(df['velocity_x'], df['velocity_y']) + 1e-6
    ux = df['geo_vector_x'] / (df['geo_distance'] + 1e-6)
    uy = df['geo_vector_y'] / (df['geo_distance'] + 1e-6)
    df['geo_alignment'] = (df['velocity_x'] * ux + df['velocity_y'] * uy) / vel_mag

    cols = [
        'time_to_endpoint',
        'geo_endpoint_x', 'geo_endpoint_y',
        'geo_vector_x', 'geo_vector_y', 'geo_distance',
        'geo_required_vx', 'geo_required_vy',
        'geo_velocity_error_x', 'geo_velocity_error_y', 'geo_velocity_error',
        'geo_required_ax', 'geo_required_ay',
        'geo_alignment',
    ]
    return df, cols


def build_features(df: pd.DataFrame, radius: float = 12.0) -> tuple[pd.DataFrame, list[str]]:
    df = ensure_basic_kinematics(df)

    df, c1 = add_role_and_ball_features(df)
    df, c2 = add_opponent_features(df, radius)
    df, c3 = add_geo_goal_features(df)
    df, c4 = add_time_features(df)
    df, c5 = add_alignment_and_change_features(df)
    df, c6 = add_field_position_features(df)

    feature_cols = sorted(set(c1 + c2 + c3 + c4 + c5 + c6))
    return df, feature_cols

# Single-Frame Token Builder

In [ ]:
def _role_to_type_id(role: str) -> int:
    return ROLE_TO_TYPE.get(role, 0)


def build_tokens_for_frame(play_df: pd.DataFrame,
                            frame_id: int,
                            player_order: list,
                            feature_cols: list) -> tuple:
    fdf = play_df[play_df['frame_id'] == frame_id]
    D = len(feature_cols)
    N_total = MAX_PLAYERS + 1

    tokens = np.zeros((N_total, D), dtype=np.float32)
    mask   = np.zeros((N_total,),   dtype=np.float32)
    ids    = np.full((N_total,), -1, dtype=np.int64)
    types  = np.zeros((N_total,),   dtype=np.int64)

    for slot, pid in enumerate(player_order[:MAX_PLAYERS]):
        row = fdf[fdf['nfl_id'] == pid]
        if len(row) == 0:
            continue

        r = row.iloc[-1]
        tokens[slot] = r[feature_cols].values.astype('float32')
        ids[slot]    = int(pid)
        mask[slot]   = 1.0

        role = r.get('player_role', 'Other')
        types[slot] = _role_to_type_id(role if isinstance(role, str) else 'Other')

    ball_vec = np.zeros((D,), dtype=np.float32)

    if {'ball_land_x', 'ball_land_y'}.issubset(fdf.columns) and len(fdf) > 0:
        ref_row = fdf.iloc[-1]
        if 'x' in feature_cols:
            ball_vec[feature_cols.index('x')] = float(ref_row.get('ball_land_x', 0.0))
        if 'y' in feature_cols:
            ball_vec[feature_cols.index('y')] = float(ref_row.get('ball_land_y', 0.0))

    tokens[-1] = ball_vec
    ids[-1]    = BALL_ID
    types[-1]  = 4
    mask[-1]   = 1.0

    return tokens, mask, ids, types

# Sequence Builder

In [ ]:
def build_target_groups(target_rows: pd.DataFrame, is_training: bool) -> pd.DataFrame:
    if is_training:
        return target_rows[GROUP_COLS].drop_duplicates().reset_index(drop=True)
    return (target_rows[GROUP_COLS + ['play_direction']]
            .drop_duplicates()
            .reset_index(drop=True))


def build_targets_for_player(target_rows: pd.DataFrame,
                              gid: int, pid: int, nfl_id: int,
                              last_x: float, last_y: float) -> tuple:
    if nfl_id == -1 or nfl_id == BALL_ID:
        return (np.array([], dtype=np.float32),
                np.array([], dtype=np.float32),
                np.array([], dtype=np.int32))

    out_grp = target_rows[
        (target_rows['game_id'] == gid) &
        (target_rows['play_id'] == pid) &
        (target_rows['nfl_id']  == nfl_id)
    ].sort_values('frame_id')

    if len(out_grp) == 0:
        return (np.array([], dtype=np.float32),
                np.array([], dtype=np.float32),
                np.array([], dtype=np.int32))

    dx = (out_grp['x'].values - last_x).astype(np.float32)
    dy = (out_grp['y'].values - last_y).astype(np.float32)
    fids = out_grp['frame_id'].values.astype(np.int32)
    return dx, dy, fids


def build_sequences(input_df: pd.DataFrame,
                     output_df: pd.DataFrame = None,
                     test_template: pd.DataFrame = None,
                     is_training: bool = True,
                     window_size: int = 10,
                     neighbor_radius: float = 12.0) -> tuple:

    print("=" * 60)
    print(f"Building sequences (training={is_training}, window={window_size})")
    print("=" * 60)

    dir_map = build_play_direction_map(input_df)
    input_u = apply_direction_to_df(input_df, dir_map)

    if is_training:
        target_rows = apply_direction_to_df(output_df, dir_map)
    else:
        target_rows = apply_direction_to_df(test_template, dir_map)

    processed_df, feature_cols = build_features(input_u, radius=neighbor_radius)

    assert 'x' in feature_cols and 'y' in feature_cols
    idx_x = feature_cols.index('x')
    idx_y = feature_cols.index('y')

    target_groups = build_target_groups(target_rows, is_training)

    sequences, masks_list, types_list = [], [], []
    targets_dx, targets_dy, targets_fids = [], [], []
    seq_meta = []

    play_cache = {}

    for row_tuple in tqdm(list(target_groups.itertuples(index=False)),
                           total=len(target_groups),
                           desc="Building sequences"):
        row = pd.Series(row_tuple, index=target_groups.columns)
        gid = row['game_id']
        pid = row['play_id']
        nid = row['nfl_id']
        play_dir = row.get('play_direction', None)

        key = (gid, pid)
        if key not in play_cache:
            pdf = (processed_df[(processed_df['game_id'] == gid) &
                                (processed_df['play_id'] == pid)]
                   .sort_values('frame_id')
                   .reset_index(drop=True))
            pdf = pdf.fillna(pdf.mean(numeric_only=True))
            play_cache[key] = {
                'df': pdf,
                'players': sorted(pdf['nfl_id'].unique().tolist()),
            }
        play_df = play_cache[key]['df']
        all_players = play_cache[key]['players']

        target_frames = play_df[play_df['nfl_id'] == nid]['frame_id'].values
        if len(target_frames) == 0:
            continue

        if len(target_frames) < window_size:
            if is_training:
                continue
            pad_len = window_size - len(target_frames)
            target_frames = np.concatenate([
                np.full(pad_len, target_frames[0], dtype=target_frames.dtype),
                target_frames,
            ])
        target_frames = target_frames[-window_size:]

        player_order = [nid] + [p for p in all_players if p != nid]
        player_order = player_order[:MAX_PLAYERS]

        T = window_size
        D = len(feature_cols)
        N_total = MAX_PLAYERS + 1

        seq    = np.zeros((T, N_total, D), dtype=np.float32)
        msk    = np.zeros((T, N_total),    dtype=np.float32)
        ttypes = np.zeros((T, N_total),    dtype=np.int64)
        last_frame_ids = None

        for t_idx, fid in enumerate(target_frames):
            tokens, m, ids, types = build_tokens_for_frame(
                play_df, fid, player_order, feature_cols
            )
            seq[t_idx] = tokens
            msk[t_idx] = m
            ttypes[t_idx] = types
            if t_idx == len(target_frames) - 1:
                last_frame_ids = ids

        sequences.append(seq)
        masks_list.append(msk)
        types_list.append(ttypes)

        if is_training:
            per_player_dx = []
            per_player_dy = []
            per_player_fids = []

            last_x_per_slot = seq[-1, :, idx_x]
            last_y_per_slot = seq[-1, :, idx_y]

            for j, slot_id in enumerate(last_frame_ids.tolist()):
                dx, dy, fids = build_targets_for_player(
                    target_rows, gid, pid, slot_id,
                    float(last_x_per_slot[j]),
                    float(last_y_per_slot[j])
                )
                per_player_dx.append(dx)
                per_player_dy.append(dy)
                per_player_fids.append(fids)

            targets_dx.append(per_player_dx)
            targets_dy.append(per_player_dy)
            targets_fids.append(per_player_fids)

        seq_meta.append({
            'game_id': int(gid),
            'play_id': int(pid),
            'nfl_id':  int(nid),
            'frame_id': int(target_frames[-1]),
            'play_direction': (None if is_training else play_dir),
            'last_frame_nfl_ids': (None if last_frame_ids is None
                                   else last_frame_ids.tolist()),
        })

    print(f"Built {len(sequences)} sequences")
    print(f"Shape per sequence: (T={window_size}, N={MAX_PLAYERS+1}, D={len(feature_cols)})")

    if is_training:
        return (sequences, masks_list, types_list,
                targets_dx, targets_dy, targets_fids,
                seq_meta, feature_cols, dir_map)
    return (sequences, masks_list, types_list,
            seq_meta, feature_cols, dir_map)

# Axial Attention Block

In [ ]:
class AxialBlock(nn.Module):
    def __init__(self, d_model=128, nhead=4, ff=512, dropout=0.1):
        super().__init__()
        self.temporal = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=ff,
            dropout=dropout, activation='gelu',
            batch_first=True, norm_first=True,
        )
        self.spatial = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=ff,
            dropout=dropout, activation='gelu',
            batch_first=True, norm_first=True,
        )
        self.edge_proj = nn.Linear(d_model, d_model)

    def forward(self, z, spatial_mask=None, edge_weights=None):
        B, T, N, H = z.shape

        z_t = z.transpose(1, 2).contiguous().view(B * N, T, H)
        z_t = self.temporal(z_t)
        z_t = z_t.view(B, N, T, H).transpose(1, 2)

        z_s = z_t.contiguous().view(B * T, N, H)

        key_padding = None
        if spatial_mask is not None:
            key_padding = (spatial_mask.view(B * T, N) == 0)

        if edge_weights is not None:
            ew = edge_weights.view(B * T, N, N)
            edge_msg = torch.bmm(ew, z_s)
            z_s = z_s + self.edge_proj(edge_msg)

        z_s = self.spatial(z_s, src_key_padding_mask=key_padding)
        z_s = z_s.view(B, T, N, H)

        return z_s

# AllPlayerSTModel

In [ ]:
class AllPlayerSTModel(nn.Module):
    def __init__(self, input_dim: int, horizon: int,
                 hidden_dim: int = 128, nheads: int = 4,
                 num_layers: int = 3, ff: int = 512, dropout: float = 0.1,
                 max_T: int = 200, max_N: int = 32, num_types: int = NUM_TYPES,
                 vel_indices: tuple = None):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.horizon = horizon
        self.vel_indices = vel_indices

        self.in_proj = nn.Linear(input_dim, hidden_dim)

        self.time_pos   = nn.Parameter(torch.randn(1, max_T, 1, hidden_dim) * 0.02)
        self.player_pos = nn.Parameter(torch.randn(1, 1, max_N, hidden_dim) * 0.02)
        self.type_emb   = nn.Embedding(num_types, hidden_dim)

        self.blocks = nn.ModuleList([
            AxialBlock(d_model=hidden_dim, nhead=nheads, ff=ff, dropout=dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(hidden_dim)

        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 256), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(256, 2 * horizon),
        )

        edge_in_dim = 2 + 1
        if vel_indices is not None:
            edge_in_dim += 2
        self.edge_mlp = nn.Sequential(
            nn.Linear(edge_in_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, 1),
        )

    def _compute_edge_weights(self, x, spatial_mask=None):
        B, T, N, D = x.shape

        raw_xy = x[..., -2:]
        dx = raw_xy.unsqueeze(3) - raw_xy.unsqueeze(2)
        dist = torch.sqrt((dx ** 2).sum(-1) + 1e-6)

        feat_list = [dx]
        if self.vel_indices is not None:
            vx_idx, vy_idx = self.vel_indices
            vel = x[..., [vx_idx, vy_idx]]
            dv = vel.unsqueeze(3) - vel.unsqueeze(2)
            feat_list.append(dv)
        feat_list.append(dist.unsqueeze(-1))

        edge_feat = torch.cat(feat_list, dim=-1)
        edge_logits = self.edge_mlp(edge_feat).squeeze(-1)

        if spatial_mask is not None:
            valid = spatial_mask.unsqueeze(3) * spatial_mask.unsqueeze(2)
            edge_logits = edge_logits.masked_fill(valid == 0, -1e4)
        else:
            valid = None

        weights = torch.softmax(edge_logits, dim=-1)
        if valid is not None:
            weights = weights * (valid > 0).float()
        return weights

    def encode(self, x, spatial_mask=None, type_ids=None):
        B, T, N, D = x.shape

        edge_weights = self._compute_edge_weights(x, spatial_mask=spatial_mask)

        h = self.in_proj(x)

        if type_ids is not None:
            te = self.type_emb(type_ids.clamp(0, self.type_emb.num_embeddings - 1))
            h = h + te

        h = h + self.time_pos[:, :T, :, :] + self.player_pos[:, :, :N, :]

        z = h
        for block in self.blocks:
            z = block(z, spatial_mask=spatial_mask, edge_weights=edge_weights)
        z = self.norm(z)
        return z

    def forward(self, x, spatial_mask=None, type_ids=None):
        z = self.encode(x, spatial_mask=spatial_mask, type_ids=type_ids)

        pooled = 0.5 * (z.mean(dim=1) + z[:, -1])

        logits = self.head(pooled)
        step = logits.view(logits.size(0), logits.size(1), self.horizon, 2).permute(0, 2, 1, 3)
        cum = torch.cumsum(step, dim=1)
        return cum

# Temporal Huber Loss with Physics Regularizers

In [ ]:
class TemporalHuber(nn.Module):
    def __init__(self, delta: float = 0.5, time_decay: float = 0.03):
        super().__init__()
        self.delta = delta
        self.time_decay = time_decay

    def forward(self, pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor,
                last_pos: torch.Tensor = None,
                lower: float = None, upper: float = None,
                step_cap: float = None,
                w_boundary: float = 0.0,
                w_speed: float = 0.0,
                w_jerk: float = 0.0) -> torch.Tensor:

        main_loss = self._huber_with_decay(pred, target, mask)

        speed_pen = self._speed_penalty(pred, mask, step_cap, w_speed)
        jerk_pen  = self._jerk_penalty(pred, mask, w_jerk)
        boundary_pen = self._boundary_penalty(pred, mask, last_pos, lower, upper, w_boundary)

        return main_loss + speed_pen + jerk_pen + boundary_pen

    def _huber_with_decay(self, pred, target, mask):
        err = pred - target
        abs_err = torch.abs(err)
        huber = torch.where(
            abs_err <= self.delta,
            0.5 * err * err,
            self.delta * (abs_err - 0.5 * self.delta),
        )

        if self.time_decay > 0:
            L = pred.size(1)
            t = torch.arange(L, device=pred.device, dtype=pred.dtype)
            w = torch.exp(-self.time_decay * t).view(1, L)
            huber = huber * w
            mask = mask * w

        return (huber * mask).sum() / (mask.sum() + 1e-8)

    def _speed_penalty(self, pred, mask, step_cap, w_speed):
        if w_speed <= 0.0 or step_cap is None:
            return 0.0

        step = torch.cat([pred[:, :1], pred[:, 1:] - pred[:, :-1]], dim=1)

        step_mask = torch.zeros_like(mask)
        step_mask[:, 0]  = mask[:, 0]
        step_mask[:, 1:] = mask[:, 1:] * mask[:, :-1]

        overflow = F.relu(torch.abs(step) - float(step_cap))
        pen = (overflow.pow(2) * step_mask).sum() / (step_mask.sum() + 1e-8)
        return w_speed * pen

    def _jerk_penalty(self, pred, mask, w_jerk):
        if w_jerk <= 0.0 or pred.size(1) <= 2:
            return 0.0

        step = torch.cat([pred[:, :1], pred[:, 1:] - pred[:, :-1]], dim=1)
        jerk = step[:, 2:] - 2 * step[:, 1:-1] + step[:, :-2]

        jerk_mask = mask[:, 2:] * mask[:, 1:-1] * mask[:, :-2]
        pen = (jerk.pow(2) * jerk_mask).sum() / (jerk_mask.sum() + 1e-8)
        return w_jerk * pen

    def _boundary_penalty(self, pred, mask, last_pos, lower, upper, w_boundary):
        if w_boundary <= 0.0 or last_pos is None or lower is None or upper is None:
            return 0.0

        abs_pos = last_pos.unsqueeze(1) + pred

        below = F.relu(float(lower) - abs_pos)
        above = F.relu(abs_pos - float(upper))

        pen = ((below.pow(2) + above.pow(2)) * mask).sum() / (mask.sum() + 1e-8)
        return w_boundary * pen

# Training Helpers + Single Fold Training

In [ ]:
def prepare_targets(batch_axis: list, max_h: int) -> tuple[torch.Tensor, torch.Tensor]:
    tensors, masks = [], []
    for arr in batch_axis:
        L = len(arr)
        padded = np.pad(arr, (0, max_h - L), constant_values=0).astype(np.float32)
        m = np.zeros(max_h, dtype=np.float32)
        m[:L] = 1.0
        tensors.append(torch.tensor(padded))
        masks.append(torch.tensor(m))
    return torch.stack(tensors), torch.stack(masks)


def prepare_targets_xy(y_dx_list: list, y_dy_list: list, max_h: int) -> tuple[torch.Tensor, torch.Tensor]:
    dx, m = prepare_targets(y_dx_list, max_h)
    dy, _ = prepare_targets(y_dy_list, max_h)
    y = torch.stack([dx, dy], dim=-1)
    return y, m


def make_plateau_scheduler(optimizer, patience: int = 5, factor: float = 0.5):
    try:
        return torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, patience=patience, factor=factor, verbose=False
        )
    except TypeError:
        return torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, patience=patience, factor=factor
        )


def train_one_fold(X_train, M_train, TI_train, y_dx_tr, y_dy_tr, lastpos_x_tr, lastpos_y_tr,
                   X_val,   M_val,   TI_val,   y_dx_va, y_dy_va, lastpos_x_va, lastpos_y_va,
                   input_dim: int, horizon: int, config,
                   feature_cols: list) -> tuple:

    device = config.DEVICE

    vx_idx = feature_cols.index('velocity_x') if 'velocity_x' in feature_cols else None
    vy_idx = feature_cols.index('velocity_y') if 'velocity_y' in feature_cols else None
    vel_indices = (vx_idx, vy_idx) if (vx_idx is not None and vy_idx is not None) else None

    model = AllPlayerSTModel(
        input_dim=input_dim,
        horizon=horizon,
        hidden_dim=config.HIDDEN_DIM,
        vel_indices=vel_indices,
    ).to(device)

    criterion = TemporalHuber(delta=0.5, time_decay=0.03)
    optimizer = torch.optim.AdamW(model.parameters(),
                                    lr=config.LEARNING_RATE,
                                    weight_decay=1e-5)
    scheduler = make_plateau_scheduler(optimizer, patience=5, factor=0.5)

    tr_batches = _build_batches(X_train, M_train, TI_train,
                                 y_dx_tr, y_dy_tr,
                                 lastpos_x_tr, lastpos_y_tr,
                                 horizon, config.BATCH_SIZE)
    va_batches = _build_batches(X_val, M_val, TI_val,
                                 y_dx_va, y_dy_va,
                                 lastpos_x_va, lastpos_y_va,
                                 horizon, config.BATCH_SIZE)

    step_cap = config.MAX_SPEED_YPS / FPS
    lower_x, upper_x = 0.0, FIELD_LENGTH
    lower_y, upper_y = 0.0, FIELD_WIDTH

    best_loss = float('inf')
    best_state = None
    bad_epochs = 0

    for epoch in range(1, config.EPOCHS + 1):
        train_loss = _run_epoch(model, criterion, optimizer, tr_batches,
                                step_cap, lower_x, upper_x, lower_y, upper_y,
                                config, device, training=True)

        val_loss = _run_epoch(model, criterion, optimizer, va_batches,
                              step_cap, lower_x, upper_x, lower_y, upper_y,
                              config, device, training=False)

        scheduler.step(val_loss)

        if epoch % 10 == 0:
            print(f"  Epoch {epoch}: train={train_loss:.4f}, val={val_loss:.4f}")

        if val_loss < best_loss:
            best_loss = val_loss
            bad_epochs = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= config.PATIENCE:
                print(f"  Early stop at epoch {epoch}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_loss


def _build_batches(X, M, TI, Ydx, Ydy, Lx, Ly, horizon, batch_size):
    batches = []
    for i in range(0, len(X), batch_size):
        end = min(i + batch_size, len(X))

        xs  = torch.tensor(np.stack(X[i:end]).astype(np.float32))
        ms  = torch.tensor(np.stack(M[i:end]).astype(np.float32))
        tis = torch.tensor(np.stack(TI[i:end]).astype(np.int64))

        ydx_batch = [Ydx[j][0] for j in range(i, end)]
        ydy_batch = [Ydy[j][0] for j in range(i, end)]
        y, mm = prepare_targets_xy(ydx_batch, ydy_batch, horizon)

        lp_x = torch.tensor(Lx[i:end, 0], dtype=torch.float32)
        lp_y = torch.tensor(Ly[i:end, 0], dtype=torch.float32)

        batches.append((xs, ms, tis, y, mm, lp_x, lp_y))
    return batches


def _run_epoch(model, criterion, optimizer, batches,
               step_cap, lower_x, upper_x, lower_y, upper_y,
               config, device, training: bool) -> float:

    model.train(training)
    losses = []

    for bx, bm_sp, bti, by, bm, blp_x, blp_y in batches:
        bx, bm_sp, bti = bx.to(device), bm_sp.to(device), bti.to(device)
        by, bm = by.to(device), bm.to(device)
        blp_x, blp_y = blp_x.to(device), blp_y.to(device)

        if training:
            pred = model(bx, spatial_mask=bm_sp, type_ids=bti)
        else:
            with torch.no_grad():
                pred = model(bx, spatial_mask=bm_sp, type_ids=bti)

        pred_target = pred[:, :, 0, :]
        pred_x = pred_target[..., 0]
        pred_y = pred_target[..., 1]

        by_x = by[..., 0]
        by_y = by[..., 1]

        loss_x = criterion(pred_x, by_x, bm,
                            last_pos=blp_x, lower=lower_x, upper=upper_x,
                            step_cap=step_cap,
                            w_boundary=config.W_BOUNDARY,
                            w_speed=config.W_SPEED,
                            w_jerk=config.W_JERK)
        loss_y = criterion(pred_y, by_y, bm,
                            last_pos=blp_y, lower=lower_y, upper=upper_y,
                            step_cap=step_cap,
                            w_boundary=config.W_BOUNDARY,
                            w_speed=config.W_SPEED,
                            w_jerk=config.W_JERK)
        loss = 0.5 * (loss_x + loss_y)

        if training:
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        losses.append(loss.item())

    return float(np.mean(losses)) if losses else float('inf')

# Cross-Validation Orchestration

In [ ]:
def augment_with_raw(X_scaled, X_raw, idx_x, idx_y, idx_off=None):
    out = []
    for s_sc, s_raw in zip(list(X_scaled), X_raw):
        raw_xy = s_raw[..., [idx_x, idx_y]].astype(np.float32)
        parts = [s_sc]
        if idx_off is not None:
            raw_off = s_raw[..., [idx_off]].astype(np.float32)
            parts.append(raw_off)
        parts.append(raw_xy)
        out.append(np.concatenate(parts, axis=-1))
    return np.stack(out)


def fit_scaler(X_train: list) -> StandardScaler:
    mats = [s.reshape(-1, s.shape[-1]) for s in X_train]
    X_fit = np.vstack(mats)
    X_fit = np.nan_to_num(X_fit, nan=0.0, posinf=0.0, neginf=0.0)

    scaler = StandardScaler().fit(X_fit)

    if hasattr(scaler, 'scale_'):
        zero_std = (scaler.scale_ == 0) | ~np.isfinite(scaler.scale_)
        if np.any(zero_std):
            scaler.scale_[zero_std] = 1.0
    return scaler


def transform_sequences(X: list, scaler: StandardScaler) -> np.ndarray:
    out = []
    for s in X:
        T, N, D = s.shape
        s_flat = s.reshape(T * N, D)
        s_clean = np.nan_to_num(s_flat, nan=0.0, posinf=0.0, neginf=0.0)
        ss = scaler.transform(s_clean).reshape(T, N, D)
        ss = np.nan_to_num(ss, nan=0.0, posinf=0.0, neginf=0.0)
        out.append(ss.astype(np.float32))
    return np.stack(out)


def evaluate_fold(model, X_val_aug, M_val, TI_val, device, batch_size):
    model.eval()
    outs = []
    with torch.no_grad():
        for i in range(0, len(X_val_aug), batch_size):
            j = min(i + batch_size, len(X_val_aug))
            X_t  = torch.tensor(X_val_aug[i:j]).to(device)
            M_t  = torch.tensor(np.stack(M_val[i:j]).astype(np.float32)).to(device)
            TI_t = torch.tensor(np.stack(TI_val[i:j]).astype(np.int64)).to(device)
            pred = model(X_t, spatial_mask=M_t, type_ids=TI_t)
            outs.append(pred.cpu().numpy())
    return np.concatenate(outs, axis=0) if outs else np.zeros((0,))


def compute_val_rmse(pred_dx, pred_dy, y_dx_list, y_dy_list, max_h):
    total_sse = 0.0
    total_n = 0
    for i in range(len(pred_dx)):
        tdx_full, m_dx = prepare_targets([y_dx_list[i]], max_h)
        tdy_full, m_dy = prepare_targets([y_dy_list[i]], max_h)
        m = (m_dx[0].cpu().numpy().astype(bool) &
             m_dy[0].cpu().numpy().astype(bool))
        tx = tdx_full[0].cpu().numpy()[m]
        ty = tdy_full[0].cpu().numpy()[m]
        px = pred_dx[i][m]
        py = pred_dy[i][m]
        total_sse += np.sum((px - tx) ** 2 + (py - ty) ** 2)
        total_n += int(m.sum())
    if total_n == 0:
        return float('nan')
    return float(np.sqrt(total_sse / (2.0 * total_n)))


def run_cv(cfg):
    print("=" * 80)
    print(f"TRAIN MODE: seeds={cfg.SEEDS}, folds={cfg.N_FOLDS}")
    print("=" * 80)

    cache_dir = cfg.OUTPUT_DIR / 'cache'
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_file = cache_dir / f'sequences_w{cfg.WINDOW_SIZE}_r{cfg.NEIGHBOR_RADIUS}.pkl'

    if cache_file.exists():
        print(f"[cache] Loading sequences from {cache_file}")
        with open(cache_file, 'rb') as f:
            cache = pickle.load(f)
        seqs         = cache['seqs']
        masks        = cache['masks']
        types        = cache['types']
        tdx          = cache['tdx']
        tdy          = cache['tdy']
        tfids        = cache['tfids']
        seq_meta     = cache['seq_meta']
        feature_cols = cache['feature_cols']
        dir_map      = cache['dir_map']
        print(f"[cache] Loaded {len(seqs)} sequences")
    else:
        train_input, train_output = _load_training_data(cfg)
        seqs, masks, types, tdx, tdy, tfids, seq_meta, feature_cols, dir_map = build_sequences(
            train_input, output_df=train_output, is_training=True,
            window_size=cfg.WINDOW_SIZE, neighbor_radius=cfg.NEIGHBOR_RADIUS,
        )
        print(f"[cache] Saving sequences to {cache_file}")
        with open(cache_file, 'wb') as f:
            pickle.dump({
                'seqs': seqs, 'masks': masks, 'types': types,
                'tdx': tdx, 'tdy': tdy, 'tfids': tfids,
                'seq_meta': seq_meta, 'feature_cols': feature_cols,
                'dir_map': dir_map,
            }, f, protocol=4)
        print(f"[cache] Saved")

    sequences  = list(seqs)
    targets_dx = list(tdx)
    targets_dy = list(tdy)
    type_ids   = list(types)

    idx_x = feature_cols.index('x')
    idx_y = feature_cols.index('y')
    idx_off = feature_cols.index('is_offense') if 'is_offense' in feature_cols else None

    last_x_all = np.array([seqs[i][-1, :, idx_x] for i in range(len(seqs))], dtype=np.float32)
    last_y_all = np.array([seqs[i][-1, :, idx_y] for i in range(len(seqs))], dtype=np.float32)

    groups = np.array([d['game_id'] for d in seq_meta])
    fold_assign_rows = []
    fold_rmse_list = []

    for seed in cfg.SEEDS:
        set_seed(seed)
        gkf = GroupKFold(n_splits=cfg.N_FOLDS)

        for fold, (tr, va) in enumerate(gkf.split(sequences, groups=groups), 1):
            print(f"\n=== Seed {seed} | Fold {fold}/{cfg.N_FOLDS} ===")

            if seed == cfg.SEEDS[0]:
                for idx in va:
                    fold_assign_rows.append({
                        'game_id': seq_meta[idx]['game_id'],
                        'play_id': seq_meta[idx]['play_id'],
                        'nfl_id':  seq_meta[idx]['nfl_id'],
                        'fold': fold,
                    })

            X_tr = [sequences[i] for i in tr]
            X_va = [sequences[i] for i in va]
            M_tr = [masks[i] for i in tr]
            M_va = [masks[i] for i in va]
            TI_tr = [type_ids[i] for i in tr]
            TI_va = [type_ids[i] for i in va]

            scaler = fit_scaler(X_tr)
            X_tr_sc = transform_sequences(X_tr, scaler)
            X_va_sc = transform_sequences(X_va, scaler)

            X_tr_aug = augment_with_raw(X_tr_sc, X_tr, idx_x, idx_y, idx_off)
            X_va_aug = augment_with_raw(X_va_sc, X_va, idx_x, idx_y, idx_off)

            model, val_loss = train_one_fold(
                X_tr_aug, M_tr, TI_tr,
                [targets_dx[i] for i in tr], [targets_dy[i] for i in tr],
                last_x_all[tr], last_y_all[tr],
                X_va_aug, M_va, TI_va,
                [targets_dx[i] for i in va], [targets_dy[i] for i in va],
                last_x_all[va], last_y_all[va],
                X_tr_aug.shape[-1], cfg.MAX_FUTURE_HORIZON, cfg,
                feature_cols=feature_cols,
            )

            pred_xy = evaluate_fold(model, X_va_aug, M_va, TI_va,
                                     cfg.DEVICE, cfg.BATCH_SIZE)
            pred_dx = pred_xy[:, :, 0, 0]
            pred_dy = pred_xy[:, :, 0, 1]

            y_va_dx = [targets_dx[i][0] for i in va]
            y_va_dy = [targets_dy[i][0] for i in va]

            rmse = compute_val_rmse(pred_dx, pred_dy, y_va_dx, y_va_dy,
                                      cfg.MAX_FUTURE_HORIZON)
            fold_rmse_list.append(rmse)
            print(f"[Fold {fold} | Seed {seed}] val={val_loss:.4f} | RMSE={rmse:.5f}")

            save_fold_artifacts(cfg, seed, fold, model, scaler)

    _print_cv_summary(fold_rmse_list)

    fold_assign_df = pd.DataFrame(fold_assign_rows) if fold_assign_rows else None
    save_meta(cfg, feature_cols, dir_map, fold_rmse_list, fold_assign=fold_assign_df)

    print(f"\nBundle written to: {cfg.MODEL_BUNDLE_DIR_TRAIN.resolve()}")


def _load_training_data(cfg) -> tuple[pd.DataFrame, pd.DataFrame]:
    data_root = Path(os.environ.get('DATA_DIR', str(cfg.DATA_DIR)))
    inp_files = sorted(data_root.rglob('train/input_2023_w*.csv'))
    out_files = sorted(data_root.rglob('train/output_2023_w*.csv'))

    print(f"[data] {len(inp_files)} input CSVs, {len(out_files)} output CSVs in {data_root}")

    if not inp_files or not out_files:
        raise FileNotFoundError(f"Training CSVs not found under {data_root}")

    train_input  = pd.concat([pd.read_csv(f) for f in inp_files], ignore_index=True)
    train_output = pd.concat([pd.read_csv(f) for f in out_files], ignore_index=True)

    train_input  = drop_bad_plays(train_input)
    train_output = drop_bad_plays(train_output)
    return train_input, train_output


def _print_cv_summary(rmse_list: list):
    print("\n" + "=" * 80)
    print("VALIDATION RMSE by fold")
    print("=" * 80)
    for i, r in enumerate(rmse_list, 1):
        print(f"  {i:02d}: RMSE = {r:.5f}")
    if rmse_list:
        mean = float(np.mean(rmse_list))
        std  = float(np.std(rmse_list))
        print("-" * 80)
        print(f"Mean RMSE: {mean:.5f}  |  Std: {std:.5f}")
    print("=" * 80)

# I/O & Bundle Management

In [ ]:
def _mkdir(p: Path):
    p.mkdir(parents=True, exist_ok=True)


def _save_json(obj, path: Path):
    with open(path, 'w') as f:
        json.dump(obj, f, indent=2)


def _load_json(path: Path):
    with open(path, 'r') as f:
        return json.load(f)


def _save_pickle(obj, path: Path):
    with open(path, 'wb') as f:
        pickle.dump(obj, f)


def _load_pickle(path: Path):
    with open(path, 'rb') as f:
        return pickle.load(f)


def ensure_bundle_dirs(cfg) -> Path:
    root = cfg.MODEL_BUNDLE_DIR_TRAIN
    _mkdir(root / 'models')
    _mkdir(root / 'scalers')
    _mkdir(root / 'meta')
    return root


def save_fold_artifacts(cfg, seed: int, fold: int,
                         model: nn.Module, scaler: StandardScaler):
    root = ensure_bundle_dirs(cfg)
    torch.save(model.state_dict(),
               root / 'models' / f'st_all_seed{seed}_fold{fold}.pth')
    _save_pickle(scaler,
                 root / 'scalers' / f'scaler_seed{seed}_fold{fold}.pkl')


def save_meta(cfg, feature_cols: list, dir_map: pd.Series,
              fold_rmse_list: list, fold_assign: pd.DataFrame = None):
    root = ensure_bundle_dirs(cfg)

    (dir_map.reset_index()
        .to_parquet(root / 'meta' / 'dir_map.parquet', index=False))

    meta = {
        'feature_cols':       feature_cols,
        'MAX_FUTURE_HORIZON': cfg.MAX_FUTURE_HORIZON,
        'WINDOW_SIZE':        cfg.WINDOW_SIZE,
        'SEEDS':              cfg.SEEDS,
        'N_FOLDS':            cfg.N_FOLDS,
        'rmse_per_fold':      fold_rmse_list,
        'model_type':         'all',
        'NUM_TYPES':          NUM_TYPES,
        'hidden_dim':         cfg.HIDDEN_DIM,
    }
    _save_json(meta, root / 'meta' / 'meta.json')

    if fold_assign is not None:
        fold_assign.to_parquet(root / 'meta' / 'train_folds.parquet', index=False)


def _parse_seed_fold(path: Path) -> tuple[int, int]:
    m = re.search(r'seed(\d+)_fold(\d+)', path.name)
    return (int(m.group(1)), int(m.group(2))) if m else (-1, -1)


def discover_bundle_for_sub(cfg) -> tuple:
    root = cfg.MODEL_BUNDLE_DIR_SUB
    assert root and root.exists(), f"Invalid MODEL_BUNDLE_DIR_SUB: {root}"

    meta = _load_json(root / 'meta' / 'meta.json')
    feat_cols = meta['feature_cols']

    model_paths  = sorted((root / 'models').glob('st_all_seed*_fold*.pth'),
                           key=_parse_seed_fold)
    scaler_paths = sorted((root / 'scalers').glob('scaler_seed*_fold*.pkl'),
                           key=_parse_seed_fold)

    assert len(model_paths) == len(scaler_paths) > 0, "bundle is incomplete"
    return root, meta, feat_cols, model_paths, scaler_paths

# I/O & Bundle Management

In [ ]:
# ============================================================
# Cell 10a: Inference - Bundle Loading & Data Pipeline
# ============================================================

_INFER_STATE = {
    'loaded': False,
    'feature_cols': None,
    'meta': None,
    'scalers': None,
    'model_paths': None,
    'models': None,
    'hidden_dim': None,
}


def load_bundle_for_inference(cfg):
    if _INFER_STATE['loaded']:
        return

    root, meta, feat_cols, model_paths, scaler_paths = discover_bundle_for_sub(cfg)

    _INFER_STATE['feature_cols'] = feat_cols
    _INFER_STATE['meta']         = meta
    _INFER_STATE['scalers']      = [_load_pickle(p) for p in scaler_paths]
    _INFER_STATE['model_paths']  = model_paths
    _INFER_STATE['models']       = [None] * len(model_paths)
    _INFER_STATE['hidden_dim']   = int(meta.get('hidden_dim', cfg.HIDDEN_DIM))
    _INFER_STATE['loaded']       = True

    print(f"[infer] Bundle loaded from {root}")
    print(f"[infer] Folds: {len(scaler_paths)}, Features: {len(feat_cols)}")


def transform_and_augment_for_inference(X_list: list,
                                          scalers: list,
                                          feature_cols: list) -> tuple[list, int]:
    idx_x = feature_cols.index('x')
    idx_y = feature_cols.index('y')
    idx_off = feature_cols.index('is_offense') if 'is_offense' in feature_cols else None

    X_aug_per_fold = []
    for scaler in scalers:
        X_scaled = transform_sequences(X_list, scaler)
        X_aug = augment_with_raw(X_scaled, X_list, idx_x, idx_y, idx_off)
        X_aug_per_fold.append(X_aug)

    if not X_aug_per_fold:
        return [], 0

    return X_aug_per_fold, X_aug_per_fold[0].shape[-1]


def ensure_models_built(cfg, input_dim_aug: int) -> list:
    feat_cols   = _INFER_STATE['feature_cols']
    meta        = _INFER_STATE['meta']
    model_paths = _INFER_STATE['model_paths']
    hidden_dim  = _INFER_STATE['hidden_dim']

    vx_idx = feat_cols.index('velocity_x') if 'velocity_x' in feat_cols else None
    vy_idx = feat_cols.index('velocity_y') if 'velocity_y' in feat_cols else None
    vel_indices = (vx_idx, vy_idx) if (vx_idx is not None and vy_idx is not None) else None

    loaded_count = 0
    for i, path in enumerate(model_paths):
        if _INFER_STATE['models'][i] is None:
            model = AllPlayerSTModel(
                input_dim=input_dim_aug,
                horizon=meta['MAX_FUTURE_HORIZON'],
                hidden_dim=hidden_dim,
                vel_indices=vel_indices,
            ).to(cfg.DEVICE)

            state = torch.load(path, map_location=cfg.DEVICE)
            model.load_state_dict(state)
            model.eval()

            _INFER_STATE['models'][i] = model
            loaded_count += 1

    if loaded_count > 0:
        print(f"[model] Loaded {loaded_count} models (input_dim={input_dim_aug})")

    return _INFER_STATE['models']

# Inference Batch + Main Orchestrator

In [ ]:
TTA_STEPS = 8
TTA_NOISE_STD = 0.015


def _add_tta_noise(X_clean: torch.Tensor, raw_tail: int, std: float) -> torch.Tensor:
    if raw_tail > 0:
        X_aug = X_clean.clone()
        X_aug[..., :-raw_tail] += torch.randn_like(X_clean[..., :-raw_tail]) * std
        return X_aug
    return X_clean + torch.randn_like(X_clean) * std


def _run_tta_inference(models: list, X_aug_per_fold: list,
                        M_t: torch.Tensor, TI_t: torch.Tensor,
                        device, raw_tail: int) -> np.ndarray:
    all_tta_preds = []

    for tta_step in range(TTA_STEPS):
        fold_preds = []
        with torch.no_grad():
            for (path, model), X_aug_clean in zip(
                    [(p, m) for p, m in zip(_INFER_STATE['model_paths'], models)],
                    X_aug_per_fold):

                X_t_clean = torch.tensor(X_aug_clean).to(device)
                X_t = X_t_clean if tta_step == 0 else _add_tta_noise(X_t_clean, raw_tail, TTA_NOISE_STD)

                pred = model(X_t, spatial_mask=M_t, type_ids=TI_t).cpu().numpy()
                fold_preds.append(pred)

        all_tta_preds.append(np.mean(fold_preds, axis=0))

    return np.mean(all_tta_preds, axis=0)


def _build_predictions_dataframe(test_df_pd: pd.DataFrame,
                                   test_meta: list,
                                   ens_pred: np.ndarray,
                                   x_last_uni: np.ndarray,
                                   y_last_uni: np.ndarray) -> pd.DataFrame:
    ens_dx = ens_pred[:, :, 0, 0]
    ens_dy = ens_pred[:, :, 0, 1]
    H = ens_dx.shape[1]

    rows = []
    tt_idx = test_df_pd.set_index(['game_id', 'play_id', 'nfl_id']).sort_index()

    for i, meta_row in enumerate(test_meta):
        gid = meta_row['game_id']
        pid = meta_row['play_id']
        nid = meta_row['nfl_id']
        play_dir = meta_row['play_direction']
        play_is_right = (play_dir == 'right')

        try:
            fids_obj = tt_idx.loc[(gid, pid, nid), 'frame_id']
            fids = fids_obj.sort_values().tolist() if isinstance(fids_obj, pd.Series) else [int(fids_obj)]
        except KeyError:
            continue

        for t, fid in enumerate(fids):
            tt = min(t, H - 1)
            x_uni = float(np.clip(x_last_uni[i] + ens_dx[i, tt], 0, FIELD_LENGTH))
            y_uni = float(np.clip(y_last_uni[i] + ens_dy[i, tt], 0, FIELD_WIDTH))
            x_out, y_out = invert_to_original_direction(x_uni, y_uni, play_is_right)
            rows.append({
                'game_id': gid, 'play_id': pid, 'nfl_id': nid, 'frame_id': int(fid),
                'x': x_out, 'y': y_out,
            })

    if not rows:
        return pd.DataFrame({'x': np.zeros(len(test_df_pd), dtype=np.float32),
                              'y': np.zeros(len(test_df_pd), dtype=np.float32)})

    pred_map = pd.DataFrame(rows)
    key_cols = ['game_id', 'play_id', 'nfl_id', 'frame_id']
    for col in key_cols:
        try:
            pred_map[col] = pred_map[col].astype(test_df_pd[col].dtype)
        except Exception:
            pass

    out = test_df_pd.merge(pred_map, on=key_cols, how='left')
    out = out.sort_values('id').reset_index(drop=True)
    out = out.fillna({'x': 0.0, 'y': 0.0})
    return out[['x', 'y']]


def infer_batch(cfg, test_df_pd: pd.DataFrame, test_input_pd: pd.DataFrame) -> pd.DataFrame:
    print(f"[batch] test={len(test_df_pd)} test_input={len(test_input_pd)}")

    test_seqs, test_masks, test_types, test_meta, feat_cols_t, _ = build_sequences(
        test_input_pd,
        test_template=test_df_pd,
        is_training=False,
        window_size=_INFER_STATE['meta']['WINDOW_SIZE'],
        neighbor_radius=cfg.NEIGHBOR_RADIUS,
    )

    assert feat_cols_t == _INFER_STATE['feature_cols'], \
        f"Feature columns mismatch.\nExpected: {_INFER_STATE['feature_cols']}\nGot: {feat_cols_t}"

    if len(test_seqs) == 0:
        print("[warn] No sequences produced; returning zeros")
        return pd.DataFrame({'x': np.zeros(len(test_df_pd), dtype=np.float32),
                              'y': np.zeros(len(test_df_pd), dtype=np.float32)})

    feat_cols = _INFER_STATE['feature_cols']
    idx_x = feat_cols.index('x')
    idx_y = feat_cols.index('y')

    X_raw = list(test_seqs)
    x_last_uni = np.array([s[-1, 0, idx_x] for s in X_raw], dtype=np.float32)
    y_last_uni = np.array([s[-1, 0, idx_y] for s in X_raw], dtype=np.float32)

    X_aug_per_fold, input_dim_aug = transform_and_augment_for_inference(
        X_raw, _INFER_STATE['scalers'], feat_cols
    )
    models = ensure_models_built(cfg, input_dim_aug)

    M_t  = torch.tensor(np.stack(test_masks).astype(np.float32)).to(cfg.DEVICE)
    TI_t = torch.tensor(np.stack(test_types).astype(np.int64)).to(cfg.DEVICE)

    raw_tail = 2 + (1 if 'is_offense' in feat_cols else 0)

    print(f"[tta] Running {TTA_STEPS} TTA steps")
    ens_pred = _run_tta_inference(models, X_aug_per_fold, M_t, TI_t, cfg.DEVICE, raw_tail)

    return _build_predictions_dataframe(test_df_pd, test_meta, ens_pred,
                                          x_last_uni, y_last_uni)


def main():
    cfg = Config()

    print(f"[main] DEVICE={cfg.DEVICE}")
    print(f"[main] DATA_DIR={cfg.DATA_DIR}")

    if TRAIN == 1:
        cfg.MODEL_BUNDLE_DIR_TRAIN = cfg.OUTPUT_DIR / 'bundle'
        run_cv(cfg)
    else:
        if not cfg.MODEL_BUNDLE_DIR_SUB.exists():
            raise FileNotFoundError(f"Bundle not found: {cfg.MODEL_BUNDLE_DIR_SUB}")

        os.environ.setdefault('MODEL_BUNDLE_DIR', str(cfg.MODEL_BUNDLE_DIR_SUB))
        load_bundle_for_inference(cfg)

        import polars as pl
        import kaggle_evaluation.nfl_inference_server

        def predict(test: pl.DataFrame, test_input: pl.DataFrame) -> pd.DataFrame:
            t0 = time.time()
            test_pd = test.to_pandas()
            test_input_pd = test_input.to_pandas()
            preds = infer_batch(cfg, test_pd, test_input_pd)
            assert len(preds) == len(test_pd), \
                f"len mismatch: preds={len(preds)} test={len(test_pd)}"
            print(f"[predict] {time.time() - t0:.2f}s "
                  f"x=[{preds['x'].min():.2f}, {preds['x'].max():.2f}] "
                  f"y=[{preds['y'].min():.2f}, {preds['y'].max():.2f}]")
            return preds

        server = kaggle_evaluation.nfl_inference_server.NFLInferenceServer(predict)
        if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
            server.serve()
        else:
            try:
                server.run_local_gateway((str(cfg.DATA_DIR),))
            except Exception as e:
                print(f"[error] Local gateway failed: {e}")


if __name__ == '__main__' or True:
    import time
    main()